# EGMS API demo

We start by setting up some imports. You can install `requests` and `jwt` as follows:

    pip3 install requests
    pip3 install pyjwt


In [1]:
import sys
import json
import time
import requests
import jwt

## Access token

First, you must obtain an API token from the CLMS website. Follow the steps outlined in the [CLMS API documentation](https://eea.github.io/clms-api-docs/authentication.html#login-in-the-clms-website). We will use this API token to obtain a short-lived access token which will grant us access to the EGMS API as well. In the example below, the token is loaded from a file called `token.jwt` located in the same directory as this notebook.

In [2]:
def get_access_token():
    service_key = json.load(open('token.jwt', 'rb'))
    private_key = service_key['private_key'].encode('utf-8')
    claim_set = {
        "iss": service_key['client_id'],
        "sub": service_key['user_id'],
        "aud": service_key['token_uri'],
        "iat": int(time.time()),
        "exp": int(time.time() + (60 * 60)),
    }
    grant = jwt.encode(claim_set, private_key, algorithm='RS256')
    result = requests.post(service_key["token_uri"], headers={ "Accept": "application/json", "Content-Type": "application/x-www-form-urlencoded" },
            data={ "grant_type": "urn:ietf:params:oauth:grant-type:jwt-bearer", "assertion": grant } )
    access_token_info_json = result.json()
    access_token = access_token_info_json.get('access_token')
    return access_token

#### Obtain access token and configure headers

We use the get_access_token() function to obtain the access token and then configure our request headers to include it. 

In [3]:
# if error you might need to install: python -m pip install cryptography


access_token = get_access_token()
headers = {"Authorization" : f"Bearer {access_token}", "Accept" : "application/json"}

## EGMS API query

Let's build some queries to the API and post them. A query is simply a JSON object containing a set of keys that filter the result set on the server side. The following keys are supported:

- `levels` : **[Required]** Only products from the supplied list of levels are returned
- `releases` : **[Required]** Only products from the specified list of release(s) are returned
- `bbox` : Only products overlapping the bounding box are returned. The bounding box is specified as a list of lon, lat coordinates:
    - `[ [min_lon, min_lat], [max_lon, max_lat] ]`.
    - The maximum size of the bounding box is 5 degrees in longitude and latitude.
    - If you specify more than two points, the bounding box is computed as the minimum and maximum longitudes and latitudes of all the supplied points. It is also possible to provide only a single point.
- `swath` : Only products with the given swath are returned.
    - Example: `IW1`
- `burstCycle` or `burst_cycle` : Only products with the given burst cycle are returned
- `relativeOrbit` or `relative_orbit` : Only products with the given relative orbit are returned
- `direction` : Only products with the given direction (ascending or descending) are returned.
- `tileId` : Only products with the given tile ID are returned.
- `productType` : Only products with the given product type are returned. This filter is usually not needed and is already provided through the `levels` key. However, it may be used to distinguish L3-datasets between `ORTHO-UP` and `ORTHO-EAST`.
- `id` : If you already have a query ID from a previous query, you may reuse the same query ID here. Note that the query ID returned by the API may change at any time. The query ID is used to download data later, and expires if it is not used to download data.

A list of valid values for levels, releases, swaths, burst cycles, relative orbits, tile IDs, product types and direction can be obtained by calling the respective API endpoints, as illustrated below.

In [4]:
#api_endpoint = "http://localhost:2008/archive"
api_endpoint = "https://egms.land.copernicus.eu/insar-api/archive"

# Get a list of levels, product releases, swaths, burst cycles, relative orbits and directions
info = { "levels" : requests.get(f"{api_endpoint}/levels", headers=headers).json(),
         "releases" : requests.get(f"{api_endpoint}/releases", headers=headers).json(),
         "swaths" : requests.get(f"{api_endpoint}/swaths", headers=headers).json(),
         "relative_orbits" : requests.get(f"{api_endpoint}/relative_orbits", headers=headers).json(),
         "bursts" : requests.get(f"{api_endpoint}/bursts", headers=headers).json(),
         "directions" : requests.get(f"{api_endpoint}/directions", headers=headers).json(),
         "tile_ids" : requests.get(f"{api_endpoint}/tile_ids", headers=headers).json(),
         "product_types" : requests.get(f"{api_endpoint}/product_types", headers=headers).json()
       }

for key, value in info.items():
    if type(value) == type({}):
        print(f"Error: {value}")
    else:
        print(f"Available {key}: {value[0:5]} {'...' if len(value) > 5 else ''}")


Available levels: ['L2A', 'L2B', 'L3'] 
Available releases: ['2019-2023', '2020-2024'] 
Available swaths: ['IW2', 'IW1', 'IW3'] 
Available relative_orbits: ['089', '066', '043', '082', '023'] ...
Available bursts: ['0171', '0167', '0168', '0164', '0169'] ...
Available directions: ['ascending', 'descending'] 
Available tile_ids: ['E40N22', 'E37N42', 'E63N15', 'E32N36', 'E29N20'] ...
Available product_types: ['BASIC', 'ORTHO-UP', 'ORTHO-EAST', 'CALIBRATED'] 


### Query types

Queries generally come in two flavors: A query with a bounding box and optional filters, or a query specifying either a burst cycle, relative orbit, swath or a level 3 tile ID. A bounding box query may additionally include any of the filters for burst cycle, relative orbit, swath, direction, tile ID or product type. Remember to always include the levels and releases filters. The filters primarily apply to L2A and L2B products. Queries for L3 products require either a tile ID or a bounding box.

In [5]:
# Let's make a query for datasets within the bounding box below.
# Note that we can supply the bounding box as a list of lon, lat pairs, and the server
# will build the bounding box based on the minimum and maximum values of the supplied points.
query = {"id": None,
         "bbox":  [
            [16.1792116578268050, 48.1162927662246176],
            [16.5896908084970605, 48.3251054142921816]
         ],
         "levels" : ["L2A"],
         "releases" : ["2020-2024"]
         }

r = requests.post(f"{api_endpoint}/search", headers=headers, data=json.dumps(query))
result = r.json()
result

{'status': True,
 'hits': [{'poly': [[15.61456687504512, 48.13816991388965],
    [15.5670333684552, 48.32632902154394],
    [16.63343111750939, 48.43955182651113],
    [16.67696764249856, 48.25142036285649]],
   'bbox': [[15.5670333684552, 48.13816991388965],
    [16.67696764249856, 48.43955182651113]],
   'path': '2020-2024/L2A/A25-146/EGMS_L2a_146_0284_IW3_VV_2020_2024_1.zip',
   'filename': 'EGMS_L2a_146_0284_IW3_VV_2020_2024_1.zip',
   'filesize': 373493017,
   'productLevel': 'L2A',
   'productType': 'BASIC',
   'burstId': '146-0284-IW3-VV',
   'direction': 'ascending',
   'relativeOrbit': '146',
   'burstCycle': '0284',
   'swath': 'IW3',
   'polarization': 'VV',
   'version': '2020_2024_1',
   'release': '2020-2024',
   'sub_release': 1},
  {'poly': [[15.65616822737365, 47.97292075599975],
    [15.60878572398691, 48.16109161972896],
    [16.67167124037768, 48.27433859647332],
    [16.71509194661104, 48.08619708826053]],
   'bbox': [[15.60878572398691, 47.97292075599975],
    [16

In [6]:
[(x["relativeOrbit"], x["filename"]) for x in result["hits"]]

[('146', 'EGMS_L2a_146_0284_IW3_VV_2020_2024_1.zip'),
 ('146', 'EGMS_L2a_146_0283_IW3_VV_2020_2024_1.zip'),
 ('022', 'EGMS_L2a_022_0786_IW1_VV_2020_2024_1.zip'),
 ('073', 'EGMS_L2a_073_0287_IW1_VV_2020_2024_1.zip'),
 ('073', 'EGMS_L2a_073_0286_IW1_VV_2020_2024_1.zip'),
 ('022', 'EGMS_L2a_022_0789_IW1_VV_2020_2024_1.zip'),
 ('124', 'EGMS_L2a_124_0789_IW2_VV_2020_2024_1.zip'),
 ('073', 'EGMS_L2a_073_0285_IW2_VV_2020_2024_1.zip'),
 ('124', 'EGMS_L2a_124_0787_IW3_VV_2020_2024_1.zip'),
 ('146', 'EGMS_L2a_146_0285_IW3_VV_2020_2024_1.zip'),
 ('124', 'EGMS_L2a_124_0787_IW2_VV_2020_2024_1.zip'),
 ('073', 'EGMS_L2a_073_0286_IW2_VV_2020_2024_1.zip'),
 ('022', 'EGMS_L2a_022_0788_IW1_VV_2020_2024_1.zip'),
 ('022', 'EGMS_L2a_022_0787_IW1_VV_2020_2024_1.zip'),
 ('124', 'EGMS_L2a_124_0788_IW2_VV_2020_2024_1.zip'),
 ('124', 'EGMS_L2a_124_0788_IW3_VV_2020_2024_1.zip'),
 ('073', 'EGMS_L2a_073_0284_IW2_VV_2020_2024_1.zip'),
 ('124', 'EGMS_L2a_124_0789_IW3_VV_2020_2024_1.zip'),
 ('073', 'EGMS_L2a_073_0285_

In [9]:
wanted_orbits = ["073"]

hits = [
    hit for hit in result["hits"]
    if (
        hit["relativeOrbit"] in wanted_orbits
    )
]

In [10]:
[(hit["relativeOrbit"], hit["filename"]) for hit in hits]

[('073', 'EGMS_L2a_073_0287_IW1_VV_2020_2024_1.zip'),
 ('073', 'EGMS_L2a_073_0286_IW1_VV_2020_2024_1.zip'),
 ('073', 'EGMS_L2a_073_0285_IW2_VV_2020_2024_1.zip'),
 ('073', 'EGMS_L2a_073_0286_IW2_VV_2020_2024_1.zip'),
 ('073', 'EGMS_L2a_073_0284_IW2_VV_2020_2024_1.zip'),
 ('073', 'EGMS_L2a_073_0285_IW1_VV_2020_2024_1.zip')]

In [11]:
hits

[{'poly': [[15.29753140574561, 48.19806204314316],
   [15.24130675945713, 48.38490029533985],
   [16.39141148694271, 48.53004665826852],
   [16.44329605156397, 48.34307180398763]],
  'bbox': [[15.24130675945713, 48.19806204314316],
   [16.44329605156397, 48.53004665826852]],
  'path': '2020-2024/L2A/A26-073/EGMS_L2a_073_0287_IW1_VV_2020_2024_1.zip',
  'filename': 'EGMS_L2a_073_0287_IW1_VV_2020_2024_1.zip',
  'filesize': 169147794,
  'productLevel': 'L2A',
  'productType': 'BASIC',
  'burstId': '073-0287-IW1-VV',
  'direction': 'ascending',
  'relativeOrbit': '073',
  'burstCycle': '0287',
  'swath': 'IW1',
  'polarization': 'VV',
  'version': '2020_2024_1',
  'release': '2020-2024',
  'sub_release': 1},
 {'poly': [[15.34698311255686, 48.03299561877828],
   [15.29098831217166, 48.21985090841257],
   [16.43725686321383, 48.36487643006297],
   [16.48894928838612, 48.17788742172885]],
  'bbox': [[15.29098831217166, 48.03299561877828],
   [16.48894928838612, 48.36487643006297]],
  'path': '

## Query results

All queries return a JSON object containing the query results. The general format of the result is as follows:

- `status` : A boolean indicating whether the query was successful or not
- `hits` : A possibly empty list containing the query results
- `id` : A string that is used when constructing download links
- `message` : *[Optional]* Typically only provided for queries that fail. May describe what you must do to make the query succeed.

### Hits

Each object in the `hits` list contains a description of a single dataset. All datasets have the following keys:

- `filename` : The filename of the dataset. This is used when generating links to download the dataset.
- `filesize` : The size of the (compressed) dataset in bytes.
- `bbox` : The bounding box of the dataset. Each item in the list is a `[lon, lat]` pair of coordinates.
- `poly` : A polygon describing the area that the dataset actually covers. The polygon always covers a smaller area than the bounding box.
- `productLevel` : The product level of the dataset (L2A, L2B or L3)
- `productType` : The product type. Currently either BASIC, CALIBRATED, ORTHO-EAST or ORTHO-UP.
- `release` : The release this dataset belongs to.
- `sub_release` : An integer that increments whenever a dataset is updated.
- `version` : The dataset version. Obtained by concatenating `release` with `sub_release`.

In addition, datasets may have some of the following properties, depending on product type: `burstId`, `relativeOrbit`, `burstCycle`, `swath`, `polarization`, `direction` and `tileId`. The `burstId` property is a combination of `relativeOrbit`, `burstCycle`, `swath` and `polarization`.

## Downloading datasets

To download a dataset identified by the query results, you must construct a link to the dataset. The link is constructed as follows:

`api_endpoint` + `/download/` + `filename` + `?id=` + `id`

The `filename` is obtained from a dataset description in the list of hits from the query result, and the `id` is obtained from the `id` field of the query result. Here's an example:

In [12]:
links = []
for hit in hits:
    link = f"{api_endpoint}/download/{hit['filename']}?id={result['id']}"
    links.append(link)
print(links)

['https://egms.land.copernicus.eu/insar-api/archive/download/EGMS_L2a_073_0287_IW1_VV_2020_2024_1.zip?id=e5898bbb897a42eeb76434d1714d1eda', 'https://egms.land.copernicus.eu/insar-api/archive/download/EGMS_L2a_073_0286_IW1_VV_2020_2024_1.zip?id=e5898bbb897a42eeb76434d1714d1eda', 'https://egms.land.copernicus.eu/insar-api/archive/download/EGMS_L2a_073_0285_IW2_VV_2020_2024_1.zip?id=e5898bbb897a42eeb76434d1714d1eda', 'https://egms.land.copernicus.eu/insar-api/archive/download/EGMS_L2a_073_0286_IW2_VV_2020_2024_1.zip?id=e5898bbb897a42eeb76434d1714d1eda', 'https://egms.land.copernicus.eu/insar-api/archive/download/EGMS_L2a_073_0284_IW2_VV_2020_2024_1.zip?id=e5898bbb897a42eeb76434d1714d1eda', 'https://egms.land.copernicus.eu/insar-api/archive/download/EGMS_L2a_073_0285_IW1_VV_2020_2024_1.zip?id=e5898bbb897a42eeb76434d1714d1eda']


The resulting links can be used to download datasets by calling an external tool such as `curl` or `wget` or by implementing your own download code in your language of choice. Note that if the query ID is idle (i.e, not used to download anything or perform additional queries), the ID will eventually expire and the query must be performed again to obtain a new query ID.

## Download the data and load them

In [14]:
from pathlib import Path

# Vienna AOI
MIN_LON = 16.179211657826805
MAX_LON = 16.58969080849706
MIN_LAT = 48.11629276622462
MAX_LAT = 48.32510541429218

# Where to store the downloaded EGMS ZIP files
DOWNLOAD_DIR = Path("/home/lukasscharf/projects/ADUCAT/data/EGMS")

DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

print(f"Download directory: {DOWNLOAD_DIR}")

Download directory: /home/lukasscharf/projects/ADUCAT/data/EGMS


Check again for the data to download

In [15]:
for hit in hits:
    print(
        hit["filename"],
        "| orbit:", hit["relativeOrbit"],
        "| burst:", hit["burstId"],
        "| size:", round(hit["filesize"] / 1e6, 1), "MB"
    )

EGMS_L2a_073_0287_IW1_VV_2020_2024_1.zip | orbit: 073 | burst: 073-0287-IW1-VV | size: 169.1 MB
EGMS_L2a_073_0286_IW1_VV_2020_2024_1.zip | orbit: 073 | burst: 073-0286-IW1-VV | size: 429.5 MB
EGMS_L2a_073_0285_IW2_VV_2020_2024_1.zip | orbit: 073 | burst: 073-0285-IW2-VV | size: 293.5 MB
EGMS_L2a_073_0286_IW2_VV_2020_2024_1.zip | orbit: 073 | burst: 073-0286-IW2-VV | size: 166.5 MB
EGMS_L2a_073_0284_IW2_VV_2020_2024_1.zip | orbit: 073 | burst: 073-0284-IW2-VV | size: 224.2 MB
EGMS_L2a_073_0285_IW1_VV_2020_2024_1.zip | orbit: 073 | burst: 073-0285-IW1-VV | size: 251.9 MB


### Actual downloading the data

In [16]:
for link in links:
    filename = link.split("/download/")[1].split("?")[0]
    output_file = DOWNLOAD_DIR / filename

    print(f"Downloading {filename}...")

    with requests.get(link, headers=headers, stream=True) as r:
        r.raise_for_status()

        with open(output_file, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)

    print(f"Saved: {output_file}")

Saved: /home/lukasscharf/projects/ADUCAT/data/EGMS/EGMS_L2a_073_0287_IW1_VV_2020_2024_1.zip
Saved: /home/lukasscharf/projects/ADUCAT/data/EGMS/EGMS_L2a_073_0286_IW1_VV_2020_2024_1.zip
Saved: /home/lukasscharf/projects/ADUCAT/data/EGMS/EGMS_L2a_073_0285_IW2_VV_2020_2024_1.zip
Saved: /home/lukasscharf/projects/ADUCAT/data/EGMS/EGMS_L2a_073_0286_IW2_VV_2020_2024_1.zip
Saved: /home/lukasscharf/projects/ADUCAT/data/EGMS/EGMS_L2a_073_0284_IW2_VV_2020_2024_1.zip
Saved: /home/lukasscharf/projects/ADUCAT/data/EGMS/EGMS_L2a_073_0285_IW1_VV_2020_2024_1.zip


## Unzip data

In [17]:
import zipfile

extract_dir = DOWNLOAD_DIR / "extracted"
extract_dir.mkdir(parents=True, exist_ok=True)

for zip_file in DOWNLOAD_DIR.glob("*.zip"):
    print(f"Extracting {zip_file.name}")

    with zipfile.ZipFile(zip_file, "r") as z:
        z.extractall(extract_dir)

Extracting EGMS_L2a_073_0286_IW1_VV_2020_2024_1.zip
Extracting EGMS_L2a_073_0285_IW1_VV_2020_2024_1.zip
Extracting EGMS_L2a_073_0286_IW2_VV_2020_2024_1.zip
Extracting EGMS_L2a_073_0285_IW2_VV_2020_2024_1.zip
Extracting EGMS_L2a_073_0287_IW1_VV_2020_2024_1.zip
Extracting EGMS_L2a_073_0284_IW2_VV_2020_2024_1.zip


In [18]:
# check the extracted files
for f in extract_dir.rglob("*"):
    if f.is_file():
        print(f)

/home/lukasscharf/projects/ADUCAT/data/EGMS/extracted/EGMS_L2a_073_0286_IW1_VV_2020_2024_1.csv
/home/lukasscharf/projects/ADUCAT/data/EGMS/extracted/EGMS_L2a_073_0286_IW2_VV_2020_2024_1.xml
/home/lukasscharf/projects/ADUCAT/data/EGMS/extracted/EGMS_L2a_073_0285_IW1_VV_2020_2024_1.xml
/home/lukasscharf/projects/ADUCAT/data/EGMS/extracted/EGMS_L2a_073_0286_IW2_VV_2020_2024_1.csv
/home/lukasscharf/projects/ADUCAT/data/EGMS/extracted/EGMS_L2a_073_0287_IW1_VV_2020_2024_1.csv
/home/lukasscharf/projects/ADUCAT/data/EGMS/extracted/EGMS_L2a_073_0285_IW1_VV_2020_2024_1.csv
/home/lukasscharf/projects/ADUCAT/data/EGMS/extracted/EGMS_L2a_073_0285_IW2_VV_2020_2024_1.csv
/home/lukasscharf/projects/ADUCAT/data/EGMS/extracted/EGMS_L2a_073_0287_IW1_VV_2020_2024_1.xml
/home/lukasscharf/projects/ADUCAT/data/EGMS/extracted/EGMS_L2a_073_0285_IW2_VV_2020_2024_1.xml
/home/lukasscharf/projects/ADUCAT/data/EGMS/extracted/EGMS_L2a_073_0284_IW2_VV_2020_2024_1.xml
/home/lukasscharf/projects/ADUCAT/data/EGMS/extrac

In [19]:
# check which EGMS files were downloaded

csv_files = list(extract_dir.rglob("*.csv"))

print(f"Found {len(csv_files)} CSV files")

for f in csv_files:
    print(f)

Found 6 CSV files
/home/lukasscharf/projects/ADUCAT/data/EGMS/extracted/EGMS_L2a_073_0286_IW1_VV_2020_2024_1.csv
/home/lukasscharf/projects/ADUCAT/data/EGMS/extracted/EGMS_L2a_073_0286_IW2_VV_2020_2024_1.csv
/home/lukasscharf/projects/ADUCAT/data/EGMS/extracted/EGMS_L2a_073_0287_IW1_VV_2020_2024_1.csv
/home/lukasscharf/projects/ADUCAT/data/EGMS/extracted/EGMS_L2a_073_0285_IW1_VV_2020_2024_1.csv
/home/lukasscharf/projects/ADUCAT/data/EGMS/extracted/EGMS_L2a_073_0285_IW2_VV_2020_2024_1.csv
/home/lukasscharf/projects/ADUCAT/data/EGMS/extracted/EGMS_L2a_073_0284_IW2_VV_2020_2024_1.csv
